### Load libraries and engine

In [1]:
#using Pkg
#Pkg.add(["Revise", "MLDatasets", "BenchmarkTools", "ProfileSVG"]) # Run once
using Revise
using Random
using MLDatasets
using BenchmarkTools
using Profile
using ProfileSVG

includet("engines/cnn_library_v9.jl")

### Config

In [2]:
Base.@kwdef mutable struct Config
    lr::Float32 = 1f-2
    batch_size::Int = 1
    dropout_p::Float32 = 0.4f0
    rand_seed::Int = 0
    bench_samples::Int = 60      # n samples in dataset
    bench_epochs::Int = 50       # Liczba iteracji dla statystyk profilera
end

cfg = Config()
Random.seed!(cfg.rand_seed)

TaskLocalRNG()

### Load data

In [3]:
train_data = MLDatasets.FashionMNIST(split=:train)
x_full = Float32.(reshape(train_data.features, size(train_data.features,1), size(train_data.features,2), 1, :))
y_full = zeros(Float32, 10, size(x_full, 4))
for (i, class) in enumerate(train_data.targets) y_full[class+1, i] = 1 end

x_mini = x_full[:, :, :, 1:cfg.bench_samples]
y_mini = y_full[:, 1:cfg.bench_samples]
println("[+] Data is ready! Training size: ", size(x_mini, 4))

[+] Data is ready! Training size: 60


### Utils functions

In [4]:
function train_bench!(model, x, y, opt, input, target, batchsize)
    N = size(x, 4)
    for batch_start in 1:batchsize:N
        batch_end = min(batch_start + batchsize - 1, N)
        for sample in batch_start:batch_end
            zerograd!(model)
            forward!(model, input => x[:, :, :, sample], target => y[:, sample])
            backward!(model)
            accumulate!(opt, model)
        end
        optimize!(opt, model)
    end
end

train_bench! (generic function with 1 method)

### Define model

In [5]:
net = chain((
    conv((3,3), 1 => 6, 1),
    maxPool((2, 2)),
    conv((3,3), 6 => 16, 1),
    maxPool((2, 2)),
    flatten(),
    dense(784 => 84, relu),
    dropout(cfg.dropout_p),
    dense(84 => 10),
))

input = tensor(28, 28, 1)
target = tensor(10)
target.data[1] = 1f0
output = net(input)
loss = lce(output, target)
model = graph(loss)
opt = GradientDescent(cfg.lr)

GradientDescent(Dict{GraphWeight, Array{Float32}}(), 0.01f0, 0)

### Benchmarks

In [6]:
@btime train_bench!($model, $x_mini, $y_mini, $opt, $input, $target, $(cfg.batch_size))

  12.355 ms (3300 allocations: 811.57 KiB)


In [7]:
Profile.clear()
@profile for _ in 1:cfg.bench_epochs
    train_bench!(model, x_mini, y_mini, opt, input, target, cfg.batch_size)
end
ProfileSVG.save("profiles/profile_v9.svg")

# History of improvement

* Liczba epok: ```1```
* Liczba próbek w zbiorze uczącym: ```60```
* Czasy dla komputer Rafał Musiatowicz z podłączoną ładowarką

| Wersja      | Czas        | Alokacja pamięci            | Opis      |
| ----------- | ----------- |-------------------|------------------|
| v0 | 64.897 ms | 889200 allocations: 148.57 MiB | Punkt wyjściowy |
| v1 | 62.486 ms | 889080 allocations: 148.56 MiB | Usunięcie funkcji barierowych poprzez otypowanie struktury GraphNode |
| v2 | 59.121 ms | 889380 allocations: 148.59 MiB | Posługiwanie się krotką zamiast wektorem dla wypłaszczonego grafu (przechodzenie przez forward!(), optimize!() itp...) |
| v3 | 56.135 ms | 887340 allocations: 108.96 MiB | Zadbanie o odpowiednie stosowanie operatora ```.``` (funkcje ```adjoin!()``` i ```primal!()```) |
| v4 | 43.477 ms | 885420 allocations: 65.84 MiB | Zastosowanie funkcji ```mul!()``` z biblioteki ```LinearAlgebra```|
| v5 | 24.025 ms | 3420 allocations: 1.24 MiB | Zastosowanie widoków (```@views```)|
| v6 | 21.727 ms | 3540 allocations: 819.12 KiB | Zmiana ```Float64``` na ```Float32``` |
| v7 | 12.740 ms | 3540 allocations: 819.11 KiB | Usunięcie slicingów tablicy, co wyeliminowało potrzebę używania widoków oraz funkcji ```vec()``` (przypisywanie pojedynczo wartości do odpowiednich indeksów wielowymiarowych tablic) |
| v8 | 12.873 ms | 3540 allocations: 819.06 KiB | Optymalne wykorzystanie cache procesora (iterowanie w pętlach we właściwej kolejności wymiarów). Minimalna zmiana w jednej pętli, która nie zmieniła dużo. Przeprowadzono również test, gdzie specjalnie zamieniono iterowanie pętli na najgorsze pod względem dostępu wartości dla procesora. Wtedy wyniki to: ```14.127 ms (3540 allocations: 818.98 KiB)```|
| v9_1 | 10.505 ms | 3540 allocations: 819.11 KiB | Dodanie ```@inbounds``` |
| v9_2 | 10.550 ms | 3540 allocations: 819.09 KiB | Dodanie ```@simd``` |